# Configure cache and environment

In [1]:
import os
import torch

# Persist downloaded weights in a writeable sdata connector directory. This prevents re-downloading the model if your Renku session resumes.
os.environ["HF_HOME"] = os.path.join(os.getcwd(), ".hf_cache")

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, GenerationConfig

# Determine if a GPU is available in your Renku resource allocation
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device.upper()}")

/layers/paketo-buildpacks_pip-install/packages/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: CPU


# Load model and tokenizer

In [3]:
# Using a lightweight, powerful small language model (SmolLM2-135M-Instruct)
# It downloads fast and fits easily into standard memory allocations.
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

print("Loading tokenizer and model (this may take a minute on first run)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map=device
)

# Create a text generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("Model loaded successfully!")

Loading tokenizer and model (this may take a minute on first run)...


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 3260.81it/s]


Model loaded successfully!


# Run local inference

In [3]:
# Format the prompt using the model's expected chat template
messages = [
    {"role": "user", "content": "Give me a 3-step recipe for a quick snack."}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
    clean_up_tokenization_spaces=False  
)

gen_config = GenerationConfig(
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    max_new_tokens=150,
    max_length=None,
    pad_token_id=tokenizer.eos_token_id
)

# Run generation
outputs = generator(
    prompt, 
    generation_config=gen_config,
    clean_up_tokenization_spaces=False  
)

# Print the generated response
generated_text = outputs[0]["generated_text"]
# Strip the prompt formatting out to just show the answer
answer = generated_text.split("<|im_start|>assistant\n")[-1]
print(answer)

Sure, here are three simple and delicious recipes for quick snack ideas:

1. **Miso Sushi**
Ingredients:
- 2 cans (14 oz) miso paste
- 2 cups mixed rice
- 1/2 cup diced green onion
- 1/2 cup mixed vegetables (like cucumber, cherry tomatoes, and bell peppers)
- 1/2 cup olive oil
- 1/2 cup soy sauce
- 1/2 cup brown sugar
- 1/2 cup honey
- Salt and pepper to taste
Instructions:
1. Cook the rice according to package instructions.
2. In a large bowl, mix the miso paste, rice, green
